# Setup

In [1]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd
import numpy as np

# helper functions
from tiktok_api_helpers import *

## API Setup

In [2]:
new_access_token = False
use_refresh_token = False

In [3]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

# Step 1: Load Creator Open IDs

In [4]:
# load all creators
df_creators_list = pd.read_excel("all_creators_sorted.xlsx", sheet_name="LIST_CREATOR", usecols=[1, 2, 25])
list_all_creators = df_creators_list['username'].tolist()

In [5]:
# Load or initialize the manifest (tracks which handles are already found)
if Path(MANIFEST_CSV).exists():
    manifest = pd.read_csv(MANIFEST_CSV)
else:
    manifest = pd.DataFrame({"handle": list_all_creators, "found": False})

# Load or initialize consolidated creator data pulled so far
if Path(CONSOLIDATED_CSV).exists():
    df_creators = pd.read_csv(CONSOLIDATED_CSV)
else:
    df_creators = pd.DataFrame()

# recheck manifest in case of failed runs
manifest["found"] = manifest["handle"].isin(set(df_creators['username']))
manifest.to_csv(MANIFEST_CSV, index=False)

In [6]:
manifest['found'].sum()

np.int64(723)

# Step 2: Extract List of existing target collaborations to avoid invitation conflicts

In [7]:
creator_open_id_list = df_creators.loc[df_creators['username'].isin(set(manifest['handle'])), 'creator_open_id'].tolist()
product_id_list = ['1734810555690551128']

In [8]:
# find all open IDs with conflicts
all_conflict_items = []

for i in range(0, len(creator_open_id_list), 50):
    batch = creator_open_id_list[i:i + 50]
    result = check_target_collaboration_conflicts(
        creator_open_id_list=batch,
        product_id_list=product_id_list,
    )

    if result.get("code") == 0:
        all_conflict_items.extend(result["data"]["conflict_items"])
    else:
        print(f"  ⚠️  Batch failed: {result}")

df_all_collab_conflicts = pd.DataFrame(all_conflict_items)

In [9]:
df_all_collab_conflicts

,creator_open_id,existing_collaboration_id,product_id
0,JYHREwAAAACOV4AEeERJl-yievH_HBsscEhMRcfgGP6DJP...,7658237276055176981,1734810555690551128
1,IKk84gAAAACOV4AEeERJl-yievH_HBssah1Kuu28mcrb-v...,7657213115356710674,1734810555690551128
2,xVkqtQAAAACOV4AEeERJl-yievH_HBsslyEFSJHV0AuB5R...,7657956260486792967,1734810555690551128
3,jr2xOgAAAACOV4AEeERJl-yievH_HBss088gEjG6bcCcTq...,7658237276055176981,1734810555690551128
4,jOs55AAAAACOV4AEeERJl-yievH_HBssuBxT4AEOM2Iu1b...,7658270288522561287,1734810555690551128
...,...,...,...
193,WkqEkgAAAACOV4AEeERJl-yievH_HBssQ1nqyAAq_l7w85...,7657213115356710674,1734810555690551128
194,2I4kcAAAAACOV4AEeERJl-yievH_HBsselLs8_C00l4dvH...,7658246754361788180,1734810555690551128
195,LyL4CQAAAACOV4AEeERJl-yievH_HBsslP3RIKLQ5uYkUu...,7658270288522561287,1734810555690551128
196,dkpS_AAAAACOV4AEeERJl-yievH_HBssedYzoa-0vDN8vB...,7658273774026311432,1734810555690551128


## Prepare shortlisted file without conflicts for batch processing

In [10]:
# merge all extracted data
df_creators_list_id_conflict = df_creators_list \
    .merge(df_creators[['username', 'creator_open_id']], how='left', on="username") \
    .merge(df_all_collab_conflicts, how='left', on="creator_open_id")

In [11]:
# set aside all new creators with open IDs but no conflicts
df_creators_list_id_new = df_creators_list_id_conflict[df_creators_list_id_conflict['creator_open_id'].notnull() & df_creators_list_id_conflict['existing_collaboration_id'].isnull()].copy()

In [12]:
# check counts
num_creators_with_conflict = df_creators_list_id_conflict['existing_collaboration_id'].notnull().sum()
num_creators_with_id = df_creators_list_id_conflict['creator_open_id'].notnull().sum()

print(f"{num_creators_with_conflict} out of {num_creators_with_id} with conflicts. {df_creators_list_id_new.shape[0]} new creator_open_id remaning for new target collaborations")

198 out of 723 with conflicts. 525 new creator_open_id remaning for new target collaborations


# Step 3: Create new Target Collaboration in batches of 50 new creators

In [13]:
# Set Default Values
message = "Hi {{user_name}}! \n\nWe'd love to have you as a Vitami affiliate. We make PMS Relief Gummies to help women get through their period with less discomfort. You'll get 20% commission plus a free sample for 1 TikTok video. Kindly accept this invite and request your sample if you're interested so we can ship it right away. Hoping to spread the word on better period care together! \u200d\u200d"
end_time = 2101132799
products = [{
    "id": '1734810555690551128',
    "target_commission_rate":2000, 
    "shop_ads_commission_rate": 300
}]
seller_contact_info = {
    "email": 'admin@vitamigummies.com'
}
free_sample_rule = {
    'has_free_sample': True,
    'is_sample_approval_exempt': True
}

## Send TEST Target Collab

In [11]:
# _, _, df_creators = run_pass(['giandrilon'], chunk_size=1, df_creators=df_creators)

[chunk_size=1] 1/1: searching ['giandrilon']
Rate limited (attempt 1/6). Waiting 11.0s...
Rate limited (attempt 2/6). Waiting 20.9s...
Rate limited (attempt 3/6). Waiting 40.7s...
Rate limited (attempt 4/6). Waiting 80.2s...


In [13]:
# df_creators[df_creators['username'] == 'giandrilon']

,avatar,avg_ec_live_uv,avg_ec_video_view_count,category_ids,creator_open_id,follower_count,gmv_range,nickname,selection_region,top_follower_demographics,username,gmv,video_gmv,live_gmv


In [16]:
# name = "TEST Target Collab 3"
# chunk = ['cpeEOQAAAACOV4AEeERJl-yievH_HBssKj9ANd-0OHXPxOsEvShxhA']

# result = create_target_collaboration(
#     name=name,
#     end_time=end_time,
#     products=products,
#     creator_user_open_ids=chunk,
#     seller_contact_info=seller_contact_info,
#     free_sample_rule=free_sample_rule,
#     message=message,
# )

In [18]:
# result

{'code': 0,
 'data': {'target_collaboration': {'id': '7661934415769765639'},
  'target_collaboration_conflicts': []},
 'message': 'Success',
 'request_id': '20260713165915132DA0A7D022F870FA90'}

## Batch process Target Collab

In [ ]:
# do first 100 first


In [ ]:
# # Batch-create target collaborations: group by batch_name, then chunk each group's creators into groups of 50 (the API's max per invitation)
# results = []
 
# for batch_name, group in df_creators_list_id_new.groupby('batch_name'):
#     creator_ids = group['creator_open_id'].tolist()
#     chunks = [creator_ids[i:i + 50] for i in range(0, len(creator_ids), 50)]
 
#     for chunk_count, chunk in enumerate(chunks, start=1):
#         name = f"{batch_name}_{chunk_count:02d}"
#         print(f"Creating '{name}' with {len(chunk)} creators...")
 
#         result = create_target_collaboration(
#             name=name,
#             end_time=end_time,
#             products=products,
#             creator_user_open_ids=chunk,
#             seller_contact_info=seller_contact_info,
#             free_sample_rule=free_sample_rule,
#             message=message,
#         )
 
#         if result.get("code") == 0:
#             collab_id = result["data"]["target_collaboration"]["id"]
#             print(f"  ✅ Created: {collab_id}")
#         else:
#             collab_id = None
#             print(f"  ⚠️  Failed: {result}")
 
#         results.append({
#             "name": name,
#             "batch_name": batch_name,
#             "chunk_count": chunk_count,
#             "num_creators": len(chunk),
#             "creator_open_ids": chunk,
#             "target_collaboration_id": collab_id,
#             "code": result.get("code"),
#             "message": result.get("message"),
#         })
 
# df_target_collaborations = pd.DataFrame(results)